In [48]:
from sklearn.model_selection import train_test_split
from Fuction import data
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import pandas as pd
import numpy as np


In [49]:
data()

Fetched: 1000 candles
Fetched: 2000 candles
Fetched: 3000 candles
Fetched: 4000 candles
Fetched: 5000 candles
Fetched: 6000 candles
Fetched: 7000 candles
Fetched: 8000 candles
Fetched: 9000 candles
Fetched: 10000 candles
Fetched: 11000 candles
Fetched: 12000 candles
Fetched: 13000 candles
Fetched: 13057 candles
⚠ No more data
✅ CSV saved successfully!


In [ ]:
def create_features_and_target(df):

    # --- Base Features ---
    df["body"] = abs(df["close"] - df["open"])
    df["upper_wick"] = df["high"] - df[["open", "close"]].max(axis=1)
    df["lower_wick"] = df[["open", "close"]].min(axis=1) - df["low"]
    df["range"] = df["high"] - df["low"]
    df["direction"] = (df["close"] > df["open"]).astype(int)

    # --- Rolling Features ---
    df["avg_body_3"] = df["body"].rolling(3).mean()
    df["avg_body_10"] = df["body"].rolling(10).mean()
    df["volatility"] = df["range"].rolling(10).std()
    df["momentum"] = df["close"] - df["close"].shift(10)
    df["volume_avg_10"] = df["volume"].rolling(10).mean()
    df["volume_spike"] = df["volume"] / df["volume_avg_10"]

    # --- Target ---
    df["next_body"] = df["body"].shift(-1)
    df["target"] = (df["next_body"] >= 1.3 * df["avg_body_3"]).astype(int)

    # --- Drop NaN ---
    df.dropna(inplace=True)

    

    return df 




In [50]:
df = pd.read_csv("data.csv")

df  = create_features_and_target(df)

In [51]:
df.columns

Index(['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
       'quote_asset_volume', 'num_trades', 'taker_buy_base', 'taker_buy_quote',
       'ignore', 'body', 'upper_wick', 'lower_wick', 'range', 'direction',
       'avg_body_3', 'avg_body_10', 'volatility', 'momentum', 'volume_avg_10',
       'volume_spike', 'next_body', 'target'],
      dtype='object')

In [52]:
feature_cols = ["body", "upper_wick", "lower_wick", "range", "direction"]

X = []
y = []

for i in range(10, len(df) - 1):
    window = df.iloc[i-10:i]

    features = []

    # --- 10 candle features ---
    for j in range(10):
        for col in feature_cols:
            features.append(window.iloc[j][col])

    # --- derived features ---
    features.extend([
        df.iloc[i]["avg_body_3"],
        df.iloc[i]["avg_body_10"],
        df.iloc[i]["volatility"],
        df.iloc[i]["momentum"],
        df.iloc[i]["volume_spike"]
    ])

    X.append(features)
    y.append(df.iloc[i]["target"])

# --- Create column names ---
columns = []

for j in range(10):
    for col in feature_cols:
        columns.append(f"{col}_{j}")

columns += [
    "avg_body_3",
    "avg_body_10",
    "volatility",
    "momentum",
    "volume_spike"
]

# --- DataFrame ---
X = pd.DataFrame(X, columns=columns)
y = pd.Series(y, name="target")

print("✅ Features shape:", X.shape)
print("✅ Target shape:", y.shape)
print("✅ Feature names sample:", X.columns[:10])

✅ Features shape: (13035, 55)
✅ Target shape: (13035,)
✅ Feature names sample: Index(['body_0', 'upper_wick_0', 'lower_wick_0', 'range_0', 'direction_0',
       'body_1', 'upper_wick_1', 'lower_wick_1', 'range_1', 'direction_1'],
      dtype='object')


In [31]:
X

,body_0,upper_wick_0,lower_wick_0,range_0,direction_0,body_1,upper_wick_1,lower_wick_1,range_1,direction_1,...,body_9,upper_wick_9,lower_wick_9,range_9,direction_9,avg_body_3,avg_body_10,volatility,momentum,volume_spike
0,0.00254,0.00030,0.00260,0.00544,0,0.00192,0.00012,0.00048,0.00252,1,...,0.00190,0.00040,0.00050,0.00280,0,0.001427,0.001845,0.001267,0.00390,1.176928
1,0.00192,0.00012,0.00048,0.00252,1,0.00206,0.00030,0.00014,0.00250,1,...,0.00017,0.00083,0.00138,0.00238,1,0.000790,0.001683,0.001276,0.00181,0.593586
2,0.00206,0.00030,0.00014,0.00250,1,0.00063,0.00039,0.00096,0.00198,0,...,0.00030,0.00131,0.00073,0.00234,0,0.000540,0.001592,0.001271,-0.00145,1.394333
3,0.00063,0.00039,0.00096,0.00198,0,0.00196,0.00023,0.00087,0.00306,1,...,0.00115,0.00004,0.00229,0.00348,0,0.000730,0.001603,0.001267,-0.00141,0.915491
4,0.00196,0.00023,0.00087,0.00306,1,0.00285,0.00041,0.00059,0.00385,1,...,0.00074,0.00037,0.00091,0.00202,0,0.001140,0.001560,0.001326,-0.00183,0.521067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549,0.00003,0.00113,0.00047,0.00163,0,0.00038,0.00115,0.00094,0.00247,1,...,0.00012,0.00081,0.00043,0.00136,1,0.000680,0.000839,0.000960,0.00384,0.947603
550,0.00038,0.00115,0.00094,0.00247,1,0.00242,0.00030,0.00051,0.00323,1,...,0.00101,0.00058,0.00000,0.00159,1,0.000583,0.000863,0.000987,0.00411,0.822338
551,0.00242,0.00030,0.00051,0.00323,1,0.00132,0.00066,0.00047,0.00245,1,...,0.00062,0.00058,0.00032,0.00152,1,0.000700,0.000668,0.000937,0.00215,0.891374
552,0.00132,0.00066,0.00047,0.00245,1,0.00105,0.00010,0.00035,0.00150,0,...,0.00047,0.00059,0.00042,0.00148,1,0.000713,0.000641,0.001022,0.00185,1.129603


In [53]:
from sklearn.model_selection import train_test_split

# 80% train, 20% test (NO shuffle)
split_index = int(len(X) * 0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

In [33]:
X.columns

Index(['body_0', 'upper_wick_0', 'lower_wick_0', 'range_0', 'direction_0',
       'body_1', 'upper_wick_1', 'lower_wick_1', 'range_1', 'direction_1',
       'body_2', 'upper_wick_2', 'lower_wick_2', 'range_2', 'direction_2',
       'body_3', 'upper_wick_3', 'lower_wick_3', 'range_3', 'direction_3',
       'body_4', 'upper_wick_4', 'lower_wick_4', 'range_4', 'direction_4',
       'body_5', 'upper_wick_5', 'lower_wick_5', 'range_5', 'direction_5',
       'body_6', 'upper_wick_6', 'lower_wick_6', 'range_6', 'direction_6',
       'body_7', 'upper_wick_7', 'lower_wick_7', 'range_7', 'direction_7',
       'body_8', 'upper_wick_8', 'lower_wick_8', 'range_8', 'direction_8',
       'body_9', 'upper_wick_9', 'lower_wick_9', 'range_9', 'direction_9',
       'avg_body_3', 'avg_body_10', 'volatility', 'momentum', 'volume_spike'],
      dtype='object')

In [54]:
from lightgbm import LGBMClassifier

model_lgb = LGBMClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    num_leaves=15,
    min_child_samples=30,
    subsample=0.7,
    colsample_bytree=0.7
)

model_lgb.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 3094, number of negative: 7334
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007630 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11495
[LightGBM] [Info] Number of data points in the train set: 10428, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.296701 -> initscore=-0.863056
[LightGBM] [Info] Start training from score -0.863056
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

LGBMClassifier(colsample_bytree=0.7, learning_rate=0.05, max_depth=3,
               min_child_samples=30, n_estimators=150, num_leaves=15,
               subsample=0.7)

In [55]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    scale_pos_weight=3   # ratio approx
)

model_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [56]:
from sklearn.metrics import classification_report

# Predictions
lgb_pred = model_lgb.predict(X_test)
xgb_pred = model_xgb.predict(X_test)
print("testing ")
print("🔹 LightGBM")
print(classification_report(y_test, lgb_pred))

print("🔹 XGBoost")
print(classification_report(y_test, xgb_pred))



from sklearn.metrics import classification_report

# Predictions
lgb_pred = model_lgb.predict(X_train)
xgb_pred = model_xgb.predict(X_train)
print('traing ')
print("🔹 LightGBM")
print(classification_report(y_train, lgb_pred))

print("🔹 XGBoost")
print(classification_report(y_train, xgb_pred))

testing 
🔹 LightGBM
              precision    recall  f1-score   support

           0       0.74      0.94      0.83      1804
           1       0.64      0.26      0.37       803

    accuracy                           0.73      2607
   macro avg       0.69      0.60      0.60      2607
weighted avg       0.71      0.73      0.68      2607

🔹 XGBoost
              precision    recall  f1-score   support

           0       0.82      0.58      0.68      1804
           1       0.43      0.71      0.54       803

    accuracy                           0.62      2607
   macro avg       0.62      0.65      0.61      2607
weighted avg       0.70      0.62      0.64      2607

traing 
🔹 LightGBM
              precision    recall  f1-score   support

           0       0.75      0.96      0.84      7334
           1       0.72      0.26      0.38      3094

    accuracy                           0.75     10428
   macro avg       0.74      0.61      0.61     10428
weighted avg       0.74  

In [37]:
from sklearn.metrics import classification_report

# Predictions
lgb_pred = model_lgb.predict(X_train)
xgb_pred = model_xgb.predict(X_train)

print("🔹 LightGBM")
print(classification_report(y_train, lgb_pred))

print("🔹 XGBoost")
print(classification_report(y_train, xgb_pred))

🔹 LightGBM
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       319
           1       1.00      1.00      1.00       124

    accuracy                           1.00       443
   macro avg       1.00      1.00      1.00       443
weighted avg       1.00      1.00      1.00       443

🔹 XGBoost
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       319
           1       1.00      1.00      1.00       124

    accuracy                           1.00       443
   macro avg       1.00      1.00      1.00       443
weighted avg       1.00      1.00      1.00       443



In [60]:
import joblib

# models save
joblib.dump(model_lgb, "lgb_model.pkl")
joblib.dump(model_xgb, "xgb_model.pkl")

# scaler (agar use kiya hai)

print("✅ Models & scaler saved")

✅ Models & scaler saved


In [61]:
import json

config = {
    "threshold": 0.4,
    "features": list(X.columns)
}

with open("config.json", "w") as f:
    json.dump(config, f)

print("✅ Config saved")

✅ Config saved
